# Problem 3 — Phase 2: TruncatedSVD + LinearSVC

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 1](hierarchical_problem3_phase1_features.ipynb) · **Next:** [Phase 3](hierarchical_problem3_phase3_by_depth.ipynb)

[`svm_tfidf_truncated_svd_linear_edge_factory`](hierarchical_classifier.py): TF-IDF → LSI-style reduction → linear SVM.

Same **`RUN_EXPENSIVE`** gate as Phase 1.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

from hierarchical_classifier import svm_tfidf_truncated_svd_linear_edge_factory
from hierarchical_summary_metrics import comparison_table, train_full_tree_and_summarize


In [ ]:
RUN_EXPENSIVE = False
phase2_specs = [
    ("SVM_SVD200", 200, svm_tfidf_truncated_svd_linear_edge_factory(max_features=MAX_FEATURES, n_components=200)),
    ("SVM_SVD500", 500, svm_tfidf_truncated_svd_linear_edge_factory(max_features=MAX_FEATURES, n_components=500)),
    ("SVM_SVD1000", 1000, svm_tfidf_truncated_svd_linear_edge_factory(max_features=MAX_FEATURES, n_components=1000)),
]

rows_p2 = []
if RUN_EXPENSIVE:
    for name, ncomp, factory in phase2_specs:
        t0 = time.perf_counter()
        _, row = train_full_tree_and_summarize(
            name, tree, mldata, factory, split=SPLIT, verbose=False, restrict_to_parent_subtree=RESTRICT,
        )
        row["wall_time_sec"] = time.perf_counter() - t0
        row["n_components"] = ncomp
        rows_p2.append(row)
else:
    print("SKIP (RUN_EXPENSIVE=False)")

df_p2 = comparison_table(rows_p2) if rows_p2 else pd.DataFrame()
if not df_p2.empty:
    display(df_p2.round(4))
    df_p2.to_csv(BASE / "problem3_phase2_results.csv", index=False)
df_p2
